# Phase C: probe_uncertainty

**Runs entirely on Google's GPU, not your laptop.** Before running: `Runtime -> Change runtime type -> T4 GPU`.

**Question:** can a linear probe on a model's internal activations tell whether the model was told "this is an evaluation" versus "this is ordinary deployment" -- the same eval-disclosing vs deployment-framed system prompts from `eval_awareness/experiment.py` (Phase B), but now looking at what's happening inside the model instead of just its final answer letter.

**Design (mirrors `benchmark_uncertainty/` and `eval_awareness/`):**
1. For 3 small open-weight instruct models and a set of TruthfulQA questions, build the prompt under both framings and extract the model's last-token hidden state at a middle layer.
2. Train a logistic-regression probe per model to classify framing (eval-aware vs deployment) from that activation, using group-K-fold cross-validation grouped by question -- so a question's both-condition activations never leak across the train/test split.
3. Build a long-format table of `(model, prompt, outcome)` where `outcome` = 1 if the probe's held-out prediction was correct.
4. Feed that straight into `clustered_bootstrap.two_way_cluster_bootstrap` -- the exact same call as Phase A/B -- to compare the naive CI on probe accuracy against the properly clustered one.

Models are all ungated on HuggingFace (no license click-through needed): `Qwen2.5-1.5B-Instruct`, `Qwen2.5-3B-Instruct`, `microsoft/Phi-3.5-mini-instruct`. All three fit in fp16 on a free T4 (16GB) loaded one at a time.

If something errors, copy the traceback back to Claude rather than debugging blind -- this notebook hasn't been run end-to-end yet, only reasoned through.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU -- set Runtime > Change runtime type > T4 GPU, then re-run")

In [ ]:
!git clone https://github.com/panditk455/mech-interp-uncertainty.git
%cd mech-interp-uncertainty
!pip install -q -r requirements.txt
!pip install -q transformers accelerate sentencepiece scikit-learn

In [ ]:
import gc
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from transformers import AutoModelForCausalLM, AutoTokenizer

from eval_awareness.experiment import (
    load_questions,
    format_prompt,
    EVAL_AWARE_SYSTEM,
    DEPLOYMENT_SYSTEM,
)
from clustered_bootstrap import rate_statistic, two_way_cluster_bootstrap

In [ ]:
# All three are ungated on HuggingFace -- no license click-through, no HF_TOKEN needed.
MODELS = [
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-3B-Instruct",
    "microsoft/Phi-3.5-mini-instruct",
]

N_QUESTIONS = 40  # fewer than Phase B's 60 -- this is 2x forward passes per model, not API calls, and 3 sequential model loads
N_SPLITS = 5       # GroupKFold splits for the probe, grouped by question so both conditions of a question stay together

In [ ]:
def extract_activations(model_id, questions):
    """Return one row per (question, condition): {prompt, condition, label, activation}."""
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.float16, output_hidden_states=True
    ).to("cuda")
    model.eval()
    layer_idx = model.config.num_hidden_layers // 2

    rows = []
    with torch.no_grad():
        for q in questions:
            user_prompt = format_prompt(q)
            for condition, system_prompt, label in [
                ("eval_aware", EVAL_AWARE_SYSTEM, 1),
                ("deployment_framed", DEPLOYMENT_SYSTEM, 0),
            ]:
                messages = [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ]
                # return_dict=True makes this always hand back a BatchEncoding
                # with .input_ids -- newer transformers versions return a
                # BatchEncoding here even with return_tensors="pt" alone, which
                # breaks passing the result straight into the model.
                encoded = tokenizer.apply_chat_template(
                    messages,
                    add_generation_prompt=True,
                    return_tensors="pt",
                    return_dict=True,
                )
                input_ids = encoded["input_ids"].to("cuda")
                out = model(input_ids)
                # hidden_states[layer_idx]: (batch, seq, hidden) -- take the last token's activation
                activation = out.hidden_states[layer_idx][0, -1, :].float().cpu().numpy()
                rows.append(
                    {
                        "prompt": q["prompt"],
                        "condition": condition,
                        "label": label,
                        "activation": activation,
                    }
                )

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return rows

In [ ]:
questions = load_questions(N_QUESTIONS)

all_activations = {}  # model_id -> list of row dicts
for model_id in MODELS:
    print(f"extracting activations: {model_id}")
    all_activations[model_id] = extract_activations(model_id, questions)
    print(f"  done: {len(all_activations[model_id])} rows")

In [ ]:
def probe_accuracy_table(model_id, rows, n_splits=N_SPLITS, seed=0):
    """Group-K-fold probe eval, grouped by question, returns per-row outcome records."""
    X = np.stack([r["activation"] for r in rows])
    y = np.array([r["label"] for r in rows])
    groups = np.array([r["prompt"] for r in rows])

    n_groups = len(set(groups))
    splits = min(n_splits, n_groups)
    gkf = GroupKFold(n_splits=splits)

    records = [None] * len(rows)
    for train_idx, test_idx in gkf.split(X, y, groups):
        scaler = StandardScaler().fit(X[train_idx])
        clf = LogisticRegression(max_iter=2000, random_state=seed)
        clf.fit(scaler.transform(X[train_idx]), y[train_idx])
        preds = clf.predict(scaler.transform(X[test_idx]))
        for pos, pred in zip(test_idx, preds):
            records[pos] = {
                "model": model_id,
                "prompt": rows[pos]["prompt"],
                "condition": rows[pos]["condition"],
                "outcome": int(pred == y[pos]),
            }
    return records

In [ ]:
import os

all_records = []
for model_id, rows in all_activations.items():
    all_records.extend(probe_accuracy_table(model_id, rows))

results_df = pd.DataFrame(all_records)
os.makedirs("probe_uncertainty/data", exist_ok=True)
results_df.to_csv("probe_uncertainty/data/probe_results.csv", index=False)
print(results_df.groupby("model")["outcome"].mean())
results_df.head()

In [ ]:
result = two_way_cluster_bootstrap(
    results_df,
    cluster1_col="model",
    cluster2_col="prompt",
    B=1000,
    statistic_fn=lambda d: rate_statistic(d, outcome_col="outcome"),
)

for method in ("row", "cluster1", "cluster2", "combined"):
    observed = result[method]["observed"]
    for stat_name, (lo, hi) in result[method]["ci"].items():
        print(f"  {method:<10} {stat_name:<10} {observed[stat_name]:.4f}  [{lo:.4f}, {hi:.4f}]")

## Get the results back to your laptop

Run the cell below, then move the downloaded `probe_results.csv` into `probe_uncertainty/data/` in your local clone and tell Claude the naive vs. combined CI numbers printed above -- no need to re-run any model locally, this notebook already did the only GPU-heavy part.

In [ ]:
from google.colab import files
files.download("probe_uncertainty/data/probe_results.csv")